# 🟩 SIMULADOR DE PROCESOS CON COLA - Sesión 17
## Estructuras de Datos y Compiladores - Global University

**Docente:** Omar Velázquez  
**Semana:** 9 | **Sesión:** 17  
**Tema:** Simulador de procesos + errores sintácticos  
**Fecha:** Noviembre 2025

---

# 🟩 1. INTRODUCCIÓN DIDÁCTICA (Explicación estilo Feynman)

## ¿Qué vamos a aprender?

Imagina que trabajas en un banco. Cuando llegas por la mañana, hay una **cola de clientes esperando ser atendidos**. El cajero atiende al primero que llegó, luego al segundo, y así sucesivamente. Esto es exactamente lo que ocurre en los sistemas operativos cuando procesan tareas.

### Las colas en sistemas operativos

Una **cola** (o queue en inglés) es una estructura de datos que sigue el principio **FIFO** (First In, First Out):
- **First In:** La primera tarea que llega es la primera en ser procesada.
- **First Out:** Cuando una tarea termina, sale del sistema.

### Ejemplo real

Supongamos que tu computadora tiene que procesar estos trabajos:
1. 🖨️ Imprimir un documento (llega a las 8:00 AM)
2. 💾 Guardar un archivo (llega a las 8:05 AM)
3. 📊 Calcular datos (llega a las 8:10 AM)

La computadora **NO puede hacer todo a la vez**, así que los coloca en una cola:
```
Inicio: [Imprimir] → [Guardar] → [Calcular] → Fin
       8:00 AM    8:05 AM   8:10 AM
```

La tarea de imprimir se procesa primero, luego guardar, y finalmente calcular.

### Objetivos de esta sesión

✅ Implementar una **simulación de procesamiento de tareas en cola (FIFO)**  
✅ Registrar **tiempos de espera y tiempos totales** de cada tarea  
✅ Identificar y documentar **errores sintácticos** comunes  
✅ Practicar **validación de datos** antes de procesar  

---

# 🟨 2. LABORATORIO – Simulador de procesos con cola FIFO

## Paso 1: Implementar la clase `Proceso`

Cada tarea que llega al sistema tiene estas características:
- `id`: Identificador único (ej: P1, P2, P3)
- `tiempo_llegada`: Cuándo llega la tarea al sistema
- `duración`: Cuánto tiempo tarda en ejecutarse

Además, calcularemos:
- `tiempo_inicio`: Cuándo comienza a ejecutarse
- `tiempo_fin`: Cuándo termina
- `tiempo_espera`: Cuánto tiempo esperó en la cola
- `tiempo_total`: Desde que llegó hasta que terminó

In [ ]:
# PASO 1: Definir la clase Proceso
# Esta clase representa una tarea que llega al sistema

class Proceso:
    """
    Representa un proceso (tarea) que llega a la cola del sistema.
    
    Atributos:
        id (str): Identificador único del proceso
        tiempo_llegada (int): Momento en que llega el proceso
        duración (int): Tiempo que tarda en ejecutarse
    """
    
    def __init__(self, id_proceso, tiempo_llegada, duracion):
        # EXPLICACIÓN: Guardamos los datos básicos del proceso
        self.id = id_proceso
        self.tiempo_llegada = tiempo_llegada
        self.duracion = duracion
        
        # EXPLICACIÓN: Inicializamos los tiempos en None (aún no se han calculado)
        self.tiempo_inicio = None
        self.tiempo_fin = None
        self.tiempo_espera = None
        self.tiempo_total = None
    
    def calcular_metricas(self, tiempo_inicio_real):
        """
        Calcula los tiempos después de que el proceso se ha ejecutado.
        
        EXPLICACIÓN: Cuando el sistema tiene disponibilidad, comienza a procesar
        la tarea. Este método calcula cuánto esperó y cuánto tiempo total tomó.
        """
        self.tiempo_inicio = tiempo_inicio_real
        self.tiempo_fin = self.tiempo_inicio + self.duracion
        self.tiempo_espera = self.tiempo_inicio - self.tiempo_llegada
        self.tiempo_total = self.tiempo_fin - self.tiempo_llegada
    
    def __str__(self):
        """
        EXPLICACIÓN: Este método permite imprimir los datos del proceso
        de forma legible.
        """
        return (f"Proceso {self.id}: "
                f"Llegada={self.tiempo_llegada}, "
                f"Duración={self.duracion}, "
                f"Inicio={self.tiempo_inicio}, "
                f"Fin={self.tiempo_fin}, "
                f"Espera={self.tiempo_espera}, "
                f"Total={self.tiempo_total}")

# PRUEBA: Creemos un proceso de ejemplo
print("✓ Clase Proceso definida correctamente\n")

p1 = Proceso("P1", tiempo_llegada=0, duracion=5)
print(f"Ejemplo de proceso creado: {p1}")

## Paso 2: Implementar el Simulador de Cola (FIFO)

In [ ]:
# PASO 2: Definir el Simulador de Cola FIFO
# Este sistema atenderá procesos en orden de llegada

from collections import deque  # deque es una estructura FIFO eficiente

class SimuladorCola:
    """
    Simula un sistema que procesa tareas en cola FIFO.
    
    EXPLICACIÓN FEYNMAN: Imagina una fila en una tienda. El primero que
    llega es el primero que se atiende. Cuando termina de ser atendido,
    se va y entra el siguiente.
    """
    
    def __init__(self):
        # La cola almacena los procesos pendientes
        self.cola = deque()
        
        # Registro de procesos ya ejecutados
        self.procesos_completados = []
        
        # Tiempo actual de la simulación
        self.tiempo_actual = 0
        
        # Contador de ciclos
        self.ciclos = 0
    
    def agregar_proceso(self, proceso):
        """
        Agrega un nuevo proceso a la cola.
        EXPLICACIÓN: Es como llegar a una fila. El proceso se coloca al final.
        """
        self.cola.append(proceso)
        print(f"  ➕ {proceso.id} llegó a la cola en tiempo={proceso.tiempo_llegada}")
        print(f"     Cola actual: {[p.id for p in self.cola]}\n")
    
    def mostrar_estado_cola(self):
        """
        EXPLICACIÓN: Muestra qué procesos están esperando en la cola.
        Es como tomar una foto de la fila en este momento.
        """
        if len(self.cola) == 0:
            print(f"     [Cola VACÍA]")
        else:
            procesos_ids = [p.id for p in self.cola]
            print(f"     [Cola: {' → '.join(procesos_ids)}]")
    
    def simular(self, lista_procesos):
        """
        Ejecuta la simulación completa.
        
        EXPLICACIÓN: La simulación funciona así:
        1. Agregamos procesos cuando llegan
        2. Si hay procesos en cola y el sistema está libre, procesamos
        3. Repetimos hasta procesar todos
        """
        print("="*70)
        print("INICIANDO SIMULACIÓN DE COLA FIFO")
        print("="*70 + "\n")
        
        # Ordenar procesos por tiempo de llegada
        lista_procesos.sort(key=lambda p: p.tiempo_llegada)
        
        # Índice del próximo proceso a llegar
        indice_llegada = 0
        
        while indice_llegada < len(lista_procesos) or len(self.cola) > 0:
            self.ciclos += 1
            print(f"🔄 CICLO {self.ciclos} | Tiempo actual: {self.tiempo_actual}")
            
            # PASO 1: Ver si llega algún proceso en este momento
            if indice_llegada < len(lista_procesos):
                if lista_procesos[indice_llegada].tiempo_llegada == self.tiempo_actual:
                    print(f"  Eventos en este ciclo:")
                    self.agregar_proceso(lista_procesos[indice_llegada])
                    indice_llegada += 1
            
            # PASO 2: Si hay procesos en cola, procesamos el primero
            if len(self.cola) > 0:
                proceso_actual = self.cola.popleft()  # Sacamos el primero (FIFO)
                print(f"  ⚙️ Procesando: {proceso_actual.id}")
                
                # Calcular métricas
                proceso_actual.calcular_metricas(self.tiempo_actual)
                
                # Avanzar el tiempo según la duración del proceso
                self.tiempo_actual += proceso_actual.duracion
                
                # Guardar proceso completado
                self.procesos_completados.append(proceso_actual)
                
                print(f"     Duración: {proceso_actual.duracion} unidades")
                print(f"     Tiempo de espera: {proceso_actual.tiempo_espera} unidades")
                print(f"     Tiempo total: {proceso_actual.tiempo_total} unidades")
            else:
                # Si no hay procesos, avanzar el tiempo
                if indice_llegada < len(lista_procesos):
                    siguiente_proceso = lista_procesos[indice_llegada]
                    salto_tiempo = siguiente_proceso.tiempo_llegada - self.tiempo_actual
                    print(f"  ⏭️ Cola vacía. Saltando {salto_tiempo} unidades de tiempo...")
                    self.tiempo_actual = siguiente_proceso.tiempo_llegada
            
            # Mostrar estado de la cola
            print(f"  Estado de la cola:")
            self.mostrar_estado_cola()
            print()
    
    def generar_reporte(self):
        """
        EXPLICACIÓN: Al final de la simulación, generamos un reporte
        con estadísticas de todos los procesos.
        """
        print("="*70)
        print("REPORTE FINAL DE LA SIMULACIÓN")
        print("="*70 + "\n")
        
        if len(self.procesos_completados) == 0:
            print("No hay procesos completados.")
            return
        
        print(f"{'Proceso':<12} {'Llegada':<10} {'Duración':<12} {'Inicio':<10} {'Fin':<10} {'Espera':<10} {'Total':<10}")
        print("-" * 70)
        
        for p in self.procesos_completados:
            print(f"{p.id:<12} {p.tiempo_llegada:<10} {p.duracion:<12} {p.tiempo_inicio:<10} {p.tiempo_fin:<10} {p.tiempo_espera:<10} {p.tiempo_total:<10}")
        
        print("\n" + "="*70)
        
        # Calcular promedios
        tiempo_espera_promedio = sum(p.tiempo_espera for p in self.procesos_completados) / len(self.procesos_completados)
        tiempo_total_promedio = sum(p.tiempo_total for p in self.procesos_completados) / len(self.procesos_completados)
        
        print(f"\n📊 ESTADÍSTICAS:")
        print(f"   • Total de procesos: {len(self.procesos_completados)}")
        print(f"   • Tiempo de espera promedio: {tiempo_espera_promedio:.2f} unidades")
        print(f"   • Tiempo total promedio: {tiempo_total_promedio:.2f} unidades")
        print(f"   • Tiempo final de simulación: {self.tiempo_actual} unidades\n")

print("✓ Clase SimuladorCola definida correctamente")

## Paso 3: Ejecutar la simulación con procesos válidos

In [ ]:
# PASO 3: Crear procesos y ejecutar la simulación

# EXPLICACIÓN: Creamos una lista de procesos que llegan en diferentes momentos
procesos_ejemplo = [
    Proceso("P1", tiempo_llegada=0, duracion=5),    # Llega primero, tarda 5 unidades
    Proceso("P2", tiempo_llegada=2, duracion=3),    # Llega en t=2, tarda 3 unidades
    Proceso("P3", tiempo_llegada=4, duracion=4),    # Llega en t=4, tarda 4 unidades
    Proceso("P4", tiempo_llegada=6, duracion=2),    # Llega en t=6, tarda 2 unidades
]

# Crear el simulador
simulador = SimuladorCola()

# Ejecutar la simulación
simulador.simular(procesos_ejemplo)

# Generar reporte
simulador.generar_reporte()

## Paso 4: Identificar y manejar ERRORES SINTÁCTICOS

En los sistemas reales, no siempre recibimos datos válidos. Aquí simulamos errores comunes:

In [ ]:
# PASO 4: Manejo de errores sintácticos
# EXPLICACIÓN: En la vida real, pueden llegar procesos con datos inválidos

def validar_proceso(id_p, tiempo_llegada, duracion):
    """
    Valida que los datos de un proceso sean correctos.
    
    EXPLICACIÓN: Antes de aceptar un proceso, verificamos que:
    1. El ID no esté vacío
    2. El tiempo de llegada no sea negativo
    3. La duración sea positiva
    
    Retorna: (es_valido: bool, mensaje_error: str)
    """
    
    errores = []
    
    # ERROR 1: ID vacío o None
    if id_p is None or id_p == "":
        errores.append("❌ ERROR SINTÁCTICO: El ID del proceso no puede estar vacío")
    
    # ERROR 2: Tiempo de llegada negativo
    if tiempo_llegada < 0:
        errores.append(f"❌ ERROR SINTÁCTICO: Tiempo de llegada negativo ({tiempo_llegada}). Debe ser >= 0")
    
    # ERROR 3: Duración no positiva
    if duracion <= 0:
        errores.append(f"❌ ERROR SINTÁCTICO: Duración inválida ({duracion}). Debe ser > 0")
    
    # ERROR 4: Tipo de datos incorrecto (tiempo_llegada debe ser número)
    try:
        int(tiempo_llegada)
        int(duracion)
    except (TypeError, ValueError):
        errores.append("❌ ERROR SINTÁCTICO: Los tiempos deben ser números enteros")
    
    if len(errores) > 0:
        return False, errores
    else:
        return True, ["✅ Proceso válido"]

print("✓ Función de validación definida\n")

# PRUEBAS: Intentamos crear procesos con errores
print("="*70)
print("PRUEBAS DE VALIDACIÓN - Detectando errores sintácticos")
print("="*70 + "\n")

# Caso 1: Proceso válido
print("[TEST 1] Proceso válido:")
valido, mensajes = validar_proceso("P1", 0, 5)
for msg in mensajes:
    print(f"  {msg}")
print()

# Caso 2: ID vacío
print("[TEST 2] ID vacío:")
valido, mensajes = validar_proceso("", 0, 5)
for msg in mensajes:
    print(f"  {msg}")
print()

# Caso 3: Tiempo de llegada negativo
print("[TEST 3] Tiempo de llegada negativo:")
valido, mensajes = validar_proceso("P2", -5, 3)
for msg in mensajes:
    print(f"  {msg}")
print()

# Caso 4: Duración cero
print("[TEST 4] Duración cero:")
valido, mensajes = validar_proceso("P3", 0, 0)
for msg in mensajes:
    print(f"  {msg}")
print()

# Caso 5: Duración negativa
print("[TEST 5] Duración negativa:")
valido, mensajes = validar_proceso("P4", 2, -4)
for msg in mensajes:
    print(f"  {msg}")
print()

# Caso 6: Múltiples errores
print("[TEST 6] Múltiples errores:")
valido, mensajes = validar_proceso(None, -3, 0)
for msg in mensajes:
    print(f"  {msg}")
print()

## Paso 5: Simulador con validación de errores

In [ ]:
# PASO 5: Simulador mejorado que valida antes de ejecutar

def simular_con_validacion(procesos_datos):
    """
    Ejecuta una simulación validando todos los procesos primero.
    
    Parámetro: procesos_datos es una lista de tuplas (id, tiempo_llegada, duración)
    """
    print("="*70)
    print("SIMULACIÓN CON VALIDACIÓN DE ERRORES")
    print("="*70 + "\n")
    
    procesos_validos = []
    procesos_invalidos = []
    
    print("📋 FASE 1: VALIDACIÓN DE PROCESOS\n")
    
    for id_p, tiempo_ll, duracion in procesos_datos:
        es_valido, mensajes = validar_proceso(id_p, tiempo_ll, duracion)
        
        if es_valido:
            procesos_validos.append(Proceso(id_p, tiempo_ll, duracion))
            print(f"  ✅ {id_p}: Válido (llegada={tiempo_ll}, duración={duracion})")
        else:
            procesos_invalidos.append((id_p, mensajes))
            print(f"  ❌ {id_p}: RECHAZADO")
            for msg in mensajes:
                print(f"     {msg}")
    
    print(f"\n📊 Resumen de validación:")
    print(f"   • Procesos válidos: {len(procesos_validos)}")
    print(f"   • Procesos inválidos: {len(procesos_invalidos)}")
    
    if len(procesos_validos) > 0:
        print("\n⚙️ FASE 2: EJECUTAR SIMULACIÓN\n")
        
        simulador = SimuladorCola()
        simulador.simular(procesos_validos)
        simulador.generar_reporte()
    else:
        print("\n⚠️ No hay procesos válidos para simular.")

# Crear una lista de procesos con algunos errores
procesos_con_errores = [
    ("P1", 0, 5),       # Válido
    ("", 2, 3),         # ERROR: ID vacío
    ("P3", -1, 4),      # ERROR: Tiempo negativo
    ("P4", 4, 0),       # ERROR: Duración cero
    ("P5", 6, 2),       # Válido
    ("P6", 8, -3),      # ERROR: Duración negativa
    ("P7", 10, 1),      # Válido
]

# Ejecutar simulación con validación
simular_con_validacion(procesos_con_errores)

---

# 🟦 3. BITÁCORA TÉCNICA

## Instrucciones para completar esta sección:

Documenta tu trabajo completando estas preguntas:

### ❓ Pregunta 1: ¿Cómo funciona la estructura FIFO?
**Tu respuesta:**

*Escribe aquí una explicación en tus propias palabras de cómo funciona el procesamiento FIFO. Incluye un ejemplo.*

---

### ❓ Pregunta 2: ¿Qué errores encontraste en la simulación y cómo los resolviste?
**Tu respuesta:**

*Documenta los errores sintácticos que encontraste (por ejemplo, el ID vacío, tiempo negativo, etc.) y explica cómo la función `validar_proceso()` los detecta.*

---

### ❓ Pregunta 3: ¿Qué diferencia hay entre tiempo de espera y tiempo total?
**Tu respuesta:**

*Explica con un ejemplo concreto la diferencia entre estos dos valores.*

---

### ❓ Pregunta 4: ¿Por qué es importante validar datos antes de procesarlos?
**Tu respuesta:**

*Reflexiona sobre las consecuencias de procesar datos inválidos en un sistema real.*

---

### ❓ Pregunta 5: ¿Qué aprendiste de este laboratorio?
**Tu respuesta:**

*Escribe 3-4 conceptos clave que aprendiste durante esta sesión.*

---

# 🟥 4. EVALUACIÓN - Problema para resolver

## 🎯 PROBLEMA: "Gestor de procesos con prioridad y control de errores"

### Descripción
Debes **modificar el simulador anterior** para agregar soporte de **prioridades** en los procesos. Las tareas con **menor número de prioridad son más importantes** y deben ejecutarse primero.

### Requisitos

1. **Extender la clase `Proceso`** para incluir un atributo `prioridad`

2. **Crear una nueva cola con prioridad** que ordene procesos por:
   - Primero: menor número de prioridad (más importante)
   - Segundo: si tienen la misma prioridad, por tiempo de llegada (FIFO)

3. **Validar datos incluyendo la prioridad:**
   - La prioridad debe estar entre 1 y 5 (donde 1 es crítico y 5 es bajo)
   - Si falta la prioridad o está fuera de rango, debe detectarse como error

4. **Simular múltiples procesos con errores:**
   - Algunos tendrán campos vacíos (None o "")
   - Otros tendrán datos inválidos (duración negativa, prioridad = 10, etc.)

5. **Implementar un algoritmo de validación robusto** que:
   - Imprima un reporte detallado de errores detectados
   - Procese solo los procesos válidos
   - Los procese en orden de prioridad

### Datos de prueba (incluyen errores)

```python
procesos_problema = [
    ("P1", 0, 4, 2),      # Válido: prioridad 2
    ("P2", 1, 3, None),   # ERROR: prioridad None
    ("P3", 2, 5, 1),      # Válido: prioridad 1 (más importante)
    ("P4", 3, 2, 10),     # ERROR: prioridad fuera de rango
    ("P5", 4, -2, 3),     # ERROR: duración negativa
    ("", 5, 1, 1),        # ERROR: ID vacío
    ("P7", 6, 3, 2),      # Válido: prioridad 2
    ("P8", 7, 2, 1),      # Válido: prioridad 1 (más importante)
    ("P9", -1, 4, 3),     # ERROR: tiempo negativo
    ("P10", 10, 1, 5),    # Válido: prioridad 5 (menos importante)
]
```

### Salida esperada

El programa debe:
1. Validar cada proceso
2. Imprimir qué procesos son válidos y cuáles tienen errores
3. Listar los errores específicos de cada proceso inválido
4. Ejecutar la simulación ordenando por PRIORIDAD primero
5. Mostrar un reporte final donde se vea claramente el orden de ejecución según prioridad

---

## 🚀 COMENZAR A RESOLVER

### Paso 1: Extender la clase Proceso con prioridad

Modifica la clase `Proceso` para incluir el atributo `prioridad`:

In [ ]:
# TODO (Estudiante): Completa esta clase
# INSTRUCCIONES:
# 1. Copia la clase Proceso de arriba
# 2. Agrega un parámetro 'prioridad' en __init__
# 3. Actualiza el método __str__ para mostrar la prioridad

class ProcesoPrioritario:
    """
    Versión mejorada de Proceso que incluye prioridad.
    
    ESTUDIANTE: Completa esta clase
    """
    
    def __init__(self, id_proceso, tiempo_llegada, duracion, prioridad):
        # TODO: Almacena los atributos incluyendo prioridad
        pass
    
    def calcular_metricas(self, tiempo_inicio_real):
        # TODO: Implementa igual que en Proceso
        pass
    
    def __str__(self):
        # TODO: Retorna una representación con prioridad
        pass

print("📝 Clase ProcesoPrioritario lista para ser completada por el estudiante")

### Paso 2: Crear función mejorada de validación

In [ ]:
# TODO (Estudiante): Completa esta función de validación

def validar_proceso_prioritario(id_p, tiempo_llegada, duracion, prioridad):
    """
    Valida un proceso que incluye prioridad.
    
    INSTRUCCIONES ESTUDIANTE:
    1. Valida todos los campos de la clase Proceso (id, tiempo, duración)
    2. Agrega validación para prioridad:
       - No puede ser None
       - Debe estar entre 1 y 5
    3. Retorna (es_valido: bool, lista_de_errores: list)
    """
    
    errores = []
    
    # TODO: Implementa las validaciones
    # Puedes reutilizar lógica de validar_proceso
    
    return len(errores) == 0, errores

print("📝 Función validar_proceso_prioritario lista para ser completada")

### Paso 3: Crear la clase SimuladorColaPrioritaria

In [ ]:
# TODO (Estudiante): Crea el simulador con prioridad

class SimuladorColaPrioritaria:
    """
    Simula procesos ordenados por PRIORIDAD (no FIFO).
    
    INSTRUCCIONES ESTUDIANTE:
    1. Usa una lista en lugar de deque
    2. Cuando agregues un proceso, insértalo en la posición correcta
    3. La posición correcta ordena por:
       a) Menor número de prioridad primero
       b) Si iguales prioridades, por tiempo de llegada
    4. En simular(), saca siempre el primero (índice 0)
    5. Genera un reporte mostrando el orden de ejecución por prioridad
    """
    
    def __init__(self):
        # TODO: Inicializa la cola y otros atributos
        pass
    
    def agregar_proceso(self, proceso):
        # TODO: Inserta proceso en posición ordenada por prioridad
        pass
    
    def simular(self, lista_procesos):
        # TODO: Similar a SimuladorCola pero respetando prioridad
        pass
    
    def generar_reporte(self):
        # TODO: Muestra en el reporte el campo 'prioridad'
        pass

print("📝 Clase SimuladorColaPrioritaria lista para ser completada")

### Paso 4: Ejecutar la simulación con validación y prioridad

In [ ]:
# TODO (Estudiante): Implementa esta función principal

def resolver_problema():
    """
    FUNCIÓN PRINCIPAL PARA RESOLVER EL PROBLEMA.
    
    INSTRUCCIONES:
    1. Define la lista procesos_problema con los datos de arriba
    2. Valida cada proceso
    3. Imprime reporte de validación
    4. Ejecuta simulación con procesos válidos
    5. Imprime reporte final
    """
    
    # TODO: Implementa aquí
    pass

print("📝 Función resolver_problema() lista para ser completada")

---

## 📋 Reporte Final del Estudiante

**Nombre del estudiante:** ________________  
**Fecha de finalización:** ________________  

### 1. Descripción de la solución implementada:

Explica brevemente cómo implementaste:
- La clase ProcesoPrioritario
- La función de validación
- El algoritmo de ordenamiento por prioridad

*Escribe tu respuesta aquí...*

---

### 2. Errores encontrados durante la implementación:

*Documenta qué errores encontraste y cómo los solucionaste...*

---

### 3. Ejemplo de salida de tu simulación:

*Copia aquí el reporte generado por tu código...*

---

### 4. Reflexión final:

¿Cuál es la diferencia entre una cola FIFO y una cola con prioridad?  
*Escribe tu respuesta...*

---

---

## 📚 Resumen de la sesión

### Conceptos clave aprendidos:

1. **Colas FIFO**: Estructura donde el primer elemento que entra es el primero en salir
2. **Procesos en sistemas operativos**: Tareas que se ejecutan secuencialmente
3. **Métricas de rendimiento**: Tiempo de espera, tiempo total, tiempo de inicio/fin
4. **Validación de datos**: Importancia de verificar datos antes de procesarlos
5. **Colas con prioridad**: Extensión donde se respeta orden de importancia
6. **Manejo de errores sintácticos**: Identificación y documentación de errores

### Competencias desarrolladas:

✅ Implementar estructuras de datos (colas)  
✅ Simular comportamiento de sistemas  
✅ Validar entrada de datos  
✅ Manejar errores y excepciones  
✅ Documentar trabajo técnico  
✅ Analizar rendimiento de algoritmos  

---

## 🎓 Próximos pasos

En la próxima sesión, estudiaremos:
- Análisis de complejidad (Big O)
- Estructuras de datos avanzadas (árboles, grafos)
- Algoritmos de compilación

---

**Documento creado para:** Global University  
**Docente:** Omar Velázquez  
**Curso:** Estructuras de Datos y Compiladores  
**Semana 9 - Sesión 17**